<h1>Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Data preparation" data-toc-modified-id="Data preparation-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Data preparation</a></span></li><li><span><a href="#Research-tasks" data-toc-modified-id="Task-2 research"><span class="toc-item-num">2&nbsp;&nbsp;</span>Task research</a></span></li><li><span><a href="#Fighting-imbalance" data-toc-modified-id="Fighting-imbalance-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Combating imbalance</a></span></li><li><span><a href="#Testing the model" data-toc-modified-id="Testing the model-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Testing the model</a></span></li>

#Customer churn

Clients began to leave Beta Bank. Every month. A little, but noticeable. Bank marketers have calculated that it is cheaper to retain current customers than to attract new ones.

It is necessary to predict whether the client will leave the bank in the near future or not. We are provided with historical data on customer behavior and termination of contracts with the bank.

Let's build a model with an extremely large *F1*-measure.

Additionally, we will measure *AUC-ROC* and compare its value with the *F1* measure.

Data source: [https://www.kaggle.com/barelydedicated/bank-customer-churn-modeling](https://www.kaggle.com/barelydedicated/bank-customer-churn-modeling)

## Data preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils import shuffle
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [143]:
data = pd.read_csv('../datasets/Churn.csv')
data.info()
display(data.head(20))
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8.0,113755.78,2,1,0,149756.71,1
6,7,15592531,Bartlett,822,France,Male,50,7.0,0.00,2,1,1,10062.80,0
7,8,15656148,Obinna,376,Germany,Female,29,4.0,115046.74,4,1,0,119346.88,1
8,9,15792365,He,501,France,Male,44,4.0,142051.07,2,0,1,74940.50,0
9,10,15592389,Hya,684,France,Male,27,2.0,134603.88,1,1,1,71725.73,0


,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,9091.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,4.997690,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.894723,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,2.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


All data was uploaded successfully. All column types are cast to the correct data type, however you may notice some gaps in the 'Tenure' column that will need to be filled in, everything else is fine. It turns out that preprocessing will not take much time and you can immediately begin splitting the samples.

Let's fill in the gaps in the *Tenure* column with zeros, since this attribute means “how many years a person has been a client of the bank,” and if this value is not specified, it means that the client has just started cooperation and the number of years is 0.

In [144]:
data['Tenure'] = data['Tenure'].fillna(0)

In fact, *RowNumber*, *CustomerID*, *Surname* are features that have absolutely no effect on the value of the target feature for logical reasons; in the first two features, all values ​​are unique, and to assume that the last name affects the result is more than strange. Let's remove them so as not to slow down the model training time.

In [145]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5.0,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10.0,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7.0,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3.0,75075.31,2,1,0,92888.52,1


Let's bring the column to normal form.

In [146]:
data.columns = data.columns.str.lower()


There will be difficulties when training the model, since not all features are numerical. Let's use One-hot-encode to encode quantitative variables. Also, let's not forget about the dummy trap.

In [147]:
data_ohe = pd.get_dummies(data, drop_first=True)
features = data_ohe.drop('exited', axis=1)
target = data_ohe['exited']

Since we don’t have a test sample, we’ll create one ourselves from the existing one.

In [148]:
features_for_split, features_test, target_for_split, target_test = train_test_split(features, target, test_size=0.2, random_state=12345, stratify=target)

features_train, features_valid, target_train, target_valid = train_test_split(features_for_split, target_for_split, test_size=0.25, random_state=12345, stratify=target_for_split)

## Problem research

We will find out in advance how balanced the classes are.

In [149]:
class_frequency = data_ohe['exited'].value_counts(normalize=True)
class_frequency

0    0.7963
1    0.2037
Name: exited, dtype: float64

Class 0 is almost 4 times more common than class 1!

Let's select hyperparameters in a loop. Let's use two loops:

<code>for depth in range(24, 30, 1):
    for trees in range(20, 101, 20):
        model = RandomForestClassifier(random_state=12345, n_estimators=trees, max_depth=depth)
        model.fit(features_train, target_train)
        predicted_valid = model.predict(features_valid)
        print(f1_score(target_valid, predicted_valid))
        print('Depth:', depth)
        print('Trees:', trees)
        print()
        if best_f1 < f1_score(target_valid, predicted_valid):
            best_f1 = f1_score(target_valid, predicted_valid)
            best_depth = depth
            best_trees = trees
            best_model = model
</code>

In [150]:
%%time
model = RandomForestClassifier(max_depth=27, n_estimators=40, random_state=12345)
model.fit(features_train, target_train)
predicted_valid = model.predict(features_valid)
print('F1 score valid:', f1_score(y_true=target_valid, y_pred=predicted_valid))

F1 score valid: 0.5674418604651164
CPU times: total: 281 ms
Wall time: 282 ms


The F1 metric is 0.56, but this is not surprising, they simply trained the model head-on without balancing the classes. It is not possible to raise F1 without using class balancing; this is a necessary step to achieve a good quality model.

## Fighting imbalance

The first step to improve our model is to standardize our features. Different features have different scatters of values, so if our learning algorithm did not think that some features are more important, we standardize them!

In [151]:
scaler = StandardScaler()
numerics = ['creditscore', 'age', 'tenure', 'balance', 'numofproducts', 'estimatedsalary']
scaler.fit(features_train[numerics])
features_test[numerics] = scaler.transform(features_test[numerics])
features_train[numerics] = scaler.transform(features_train[numerics])
features_valid[numerics] = scaler.transform(features_valid[numerics])

Since class 1 occurs 4 times less often than class 0, we will use the **upsampling** technique, increasing the sample.

In [152]:
def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]

    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)

    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345)
    return features_upsampled, target_upsampled

Let's get updated signs.

In [153]:
features_upsampled, target_upsampled = upsample(features_train, target_train, 4)

Let's select hyperparameters in the loop. We train the model using an enlarged sample using the **upsampling** technique.

In [0]:
%%time

best_f1 = 0
best_trees = 0
best_depth = 0
best_model = None

for depth in range(1, 30, 1):
    for trees in range(20, 200, 20):
        model = RandomForestClassifier(random_state=12345, n_estimators=trees, max_depth=depth)
        model.fit(features_upsampled, target_upsampled)
        predicted_valid = model.predict(features_valid)
        print(f1_score(target_valid, predicted_valid))
        print('Depth:', depth)
        print('Trees:', trees)
        print()
        if best_f1 < f1_score(target_valid, predicted_valid):
            best_f1 = f1_score(target_valid, predicted_valid)
            best_depth = depth
            best_trees = trees
            best_model = model

In [154]:
best_depth = 12
best_trees = 140
best_model = RandomForestClassifier(random_state=12345, max_depth=best_depth, n_estimators=best_trees)
best_model.fit(features_upsampled, target_upsampled)
predicted_valid = best_model.predict(features_valid)
best_f1 = f1_score(y_true=target_valid, y_pred=predicted_valid)
# Best model with f1 0.6371257485029941, depth 12, trees 140
# CPU times: total: 5min 2s
# Wall time: 5min 3s
print(f'Best model with f1 {best_f1}, depth {best_depth}, trees {best_trees}')

Best model with f1 0.6371257485029941, depth 12, trees 140
CPU times: total: 1.22 s
Wall time: 1.29 s


Now the F1 result on the validation set is already 0.63! Excellent result.

Let's build an ROC curve and find out how things are going there.

In [155]:
probabilities_valid = best_model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]
fpr, tpr, thresholds = roc_curve(target_valid, probabilities_one_valid)
plt.figure();

plt.plot(fpr, tpr);
plt.xlim([0.0, 1.0]);
plt.ylim([0.0, 1.0]);
plt.xlabel('False Positive Rate');
plt.ylabel('True Positive Rate');
plt.title('ROC curve');
plt.show()
auc_roc = roc_auc_score(target_valid, probabilities_one_valid)
print('ROC-AUC:', auc_roc)

From the graph you can see that our model is much better than random, and also the area of ​​0.86 tells us that our model is good enough, but it is still quite far from unity.

We've sorted out the technique of increasing the sample. Let's try an alternative: reduce the sample!

In [156]:
def downsample(features, target, fraction):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    
    features_downsampled = pd.concat([features_zeros.sample(frac=fraction, random_state=12345)] + [features_ones])
    target_downsampled = pd.concat([target_zeros.sample(frac=fraction, random_state=12345)] + [target_ones])
    
    features_downsampled, target_downsampled = shuffle(
    features_downsampled, target_downsampled, random_state=12345
    )
    return features_downsampled, target_downsampled

In [157]:
features_downsampled, target_downsampled = downsample(features_train, target_train, 0.25)

In [158]:
best_depth = 12
best_trees = 140
best_model_down = RandomForestClassifier(random_state=12345, max_depth=best_depth, n_estimators=best_trees)
best_model_down.fit(features_downsampled, target_downsampled)
predicted_valid = best_model_down.predict(features_valid)
best_f1 = f1_score(y_true=target_valid, y_pred=predicted_valid)
print(f'Best model with f1 {best_f1}, depth {best_depth}, trees {best_trees}')

Best model with f1 0.5967894239848914, depth 12, trees 140


The F1 metric value has become lower, most likely due to a decrease in the sample size and the model was not sufficiently trained.

And again we will construct the ROC curve and calculate the area under it.

In [159]:
probabilities_valid = best_model_down.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]

fpr, tpr, thresholds = roc_curve(target_valid, probabilities_one_valid)
plt.figure();

plt.plot(fpr, tpr);
plt.xlim([0.0, 1.0]);
plt.ylim([0.0, 1.0]);
plt.xlabel('False Positive Rate');
plt.ylabel('True Positive Rate');
plt.title('ROC curve');
plt.show()
auc_roc = roc_auc_score(target_valid, probabilities_one_valid)
print('ROC-AUC:', auc_roc)

The area became slightly larger than with the sampling increase technique.

## Model testing

Let's test our model.

In [160]:
predicted_test = best_model.predict(features_test)

print('F1 score on the test set:', f1_score(y_true=target_test, y_pred=predicted_test))

F1 score on the test set: 0.6135831381733021


In [161]:
probabilities_test = best_model.predict_proba(features_test)
probabilities_one_test = probabilities_test[:, 1]

fpr, tpr, thresholds = roc_curve(target_test, probabilities_one_test)
plt.figure();

plt.plot(fpr, tpr);
plt.xlim([0.0, 1.0]);
plt.ylim([0.0, 1.0]);
plt.xlabel('False Positive Rate');
plt.ylabel('True Positive Rate');
plt.title('ROC curve');
plt.show()
auc_roc = roc_auc_score(target_test, probabilities_one_test)
print('ROC-AUC:', auc_roc)

We built the ROC curve again. Here the result is quite a bit better.

Now, after our models are trained, all are checked on the test set, and the validation set has done its job, we will combine the training and validation sets, and we will train the models with our selected hyperparameters on this increased sample.

In [162]:
features_full_train = pd.concat([features_upsampled, features_valid])
target_full_train = pd.concat([target_upsampled, target_valid])
model = RandomForestClassifier(random_state=12345, max_depth=12, n_estimators=140)
model.fit(features_full_train, target_full_train)
predicted_full_test = model.predict(features_test)
print('F1 score on the enlarged test sample:', f1_score(y_true=target_test, y_pred=predicted_full_test))

F1 score on the enlarged test sample: 0.6250000000000001


The result has improved slightly. Super!

# Conclusion

The work done included loading and studying the data, given the lack of a test sample, it was necessary to correctly split the data for effective training of models. There was also a sign with blanks that had to be filled in. Since our dataset had quantitative features, we had to use One-hot-encode to convert them into numerical ones. Several random forest models were trained, each of which was accompanied by ROC curve plots, area calculations, and F1 metric results. Before training began, features that did not affect the target feature were discarded. The imbalance of classes of the target trait was considered, and as it turned out, one class was found 4 times more often than the other! However, the model was trained without taking into account class imbalance, the resulting F1 metric was 0.56, which cannot be called a good result, and only then trained taking into account the imbalance, using the **upsampling** technique, as well as scaling all numerical features, after which the F1 metric results improved on the validation set, and then our model performed well on the test set. Also, in the hope of improving the result, the training and test samples were combined, and then the model was trained again, the result became better.